In [1]:
# Étape 18 — Préparer le référentiel des titres
import pandas as pd

referentiel = pd.DataFrame(
    [
        ("AMD", "US0079031078", "Nasdaq Global Select Market", "0000002488"),
        ("NVDA", None, None, None),
        ("INTC", None, None, None),
        ("AVGO", None, None, None),
        ("MRVL", None, None, None),
    ],
    columns=["Ticker", "ISIN", "Place_cotation", "CIK"],
)

display(referentiel)

,Ticker,ISIN,Place_cotation,CIK
0,AMD,US0079031078,Nasdaq Global Select Market,0000002488
1,NVDA,None,None,None
2,INTC,None,None,None
3,AVGO,None,None,None
4,MRVL,None,None,None


In [2]:
# Étape 19 — Compléter les CIK des cinq actions
import requests
from getpass import getpass

contact_sec = getpass("Adresse e-mail de contact pour la SEC : ")

reponse = requests.get(
    "https://www.sec.gov/files/company_tickers.json",
    headers={"User-Agent": f"Select Equity {contact_sec}"},
    timeout=30,
)
reponse.raise_for_status()

cik_par_ticker = {
    ligne["ticker"]: str(ligne["cik_str"]).zfill(10)
    for ligne in reponse.json().values()
}

cik_recuperes = referentiel["Ticker"].map(cik_par_ticker)
if cik_recuperes.isna().any() or cik_par_ticker.get("AMD") != "0000002488":
    raise ValueError("Vérifie les tickers et le CIK AMD avant de continuer.")

referentiel["CIK"] = cik_recuperes
display(referentiel[["Ticker", "CIK"]])

Adresse e-mail de contact pour la SEC :  ········


,Ticker,CIK
0,AMD,0000002488
1,NVDA,0001045810
2,INTC,0000050863
3,AVGO,0001730168
4,MRVL,0001835632


In [3]:
#Étape 20 — Renseigner les ISIN
isin_par_ticker = {
    "AMD": "US0079031078",
    "NVDA": "US67066G1040",
    "INTC": "US4581401001",
    "AVGO": "US11135F1012",
    "MRVL": "US5738741041",
}

referentiel["ISIN"] = referentiel["Ticker"].map(isin_par_ticker)
display(referentiel[["Ticker", "ISIN", "CIK"]])


,Ticker,ISIN,CIK
0,AMD,US0079031078,0000002488
1,NVDA,US67066G1040,0001045810
2,INTC,US4581401001,0000050863
3,AVGO,US11135F1012,0001730168
4,MRVL,US5738741041,0001835632


In [4]:
# Étape 21 — Compléter la place de cotation
referentiel["Place_cotation"] = "Nasdaq"
display(referentiel)

,Ticker,ISIN,Place_cotation,CIK
0,AMD,US0079031078,Nasdaq,0000002488
1,NVDA,US67066G1040,Nasdaq,0001045810
2,INTC,US4581401001,Nasdaq,0000050863
3,AVGO,US11135F1012,Nasdaq,0001730168
4,MRVL,US5738741041,Nasdaq,0001835632


In [5]:
# Étape 22 — Enregistrer le référentiel pour SQL
from pathlib import Path

fichier_referentiel = Path("data/processed/referentiel_titres.csv")
fichier_referentiel.parent.mkdir(parents=True, exist_ok=True)

referentiel.to_csv(fichier_referentiel, index=False)

controle = pd.read_csv(fichier_referentiel, dtype={"CIK": str})
print("Fichier :", fichier_referentiel)
print("Lignes relues :", len(controle))
display(controle)


Fichier : data/processed/referentiel_titres.csv
Lignes relues : 5


,Ticker,ISIN,Place_cotation,CIK
0,AMD,US0079031078,Nasdaq,0000002488
1,NVDA,US67066G1040,Nasdaq,0001045810
2,INTC,US4581401001,Nasdaq,0000050863
3,AVGO,US11135F1012,Nasdaq,0001730168
4,MRVL,US5738741041,Nasdaq,0001835632
